# 仮説
ここに今回の仮説を 1-3 行で書く。

# 変更点（ファイル/パラメータ）
- sweep YAML: `notebooks/experiments/sweeps/`
- 何を比較するか（ベースモデル / データ / 方策 / ハイパラ）

# 成功条件
どの数値や振る舞いをもって成功とするか。

# 結果（後で追記）
pass / fail / 数値 / 気づき

# 次
次に試す 1 手を書く。


In [ ]:
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd().resolve()
if NOTEBOOKS_DIR.name == "experiments":
    NOTEBOOKS_DIR = NOTEBOOKS_DIR.parent
elif NOTEBOOKS_DIR.name != "notebooks":
    candidate = NOTEBOOKS_DIR / "notebooks"
    NOTEBOOKS_DIR = candidate if candidate.is_dir() else NOTEBOOKS_DIR

LIB_DIR = NOTEBOOKS_DIR / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

from exp_orchestrator.expand import load_sweep, expand_lifelong_trials
from exp_orchestrator.parc_jobs import enqueue_parc_sweep, pause_job, resume_run, queue_status
from exp_orchestrator.collect import collect_all, print_ranking, print_status
from exp_orchestrator.lifelong_jobs import run_lifelong_sweep, pause_lifelong_trial

SWEEP = NOTEBOOKS_DIR / "experiments" / "sweeps" / "smolvla_lora_r_steps.yaml"
sweep = load_sweep(SWEEP)
sweep["name"], sweep["kind"], sweep.get("search")


## 投入（SmolVLA / parc）

`parc-worker` が別プロセスで動いている必要があります。未起動なら次をリポジトリの `parc/` で実行します。

```bash
cd parc && uv run parc-worker --loop --poll-sec 30
```


In [ ]:
# kind=parc のときだけ enqueue。lifelong は下のセルを使う。
if sweep.get("kind") == "parc":
    job_ids = enqueue_parc_sweep(SWEEP)
    print(job_ids)
else:
    print("this sweep is lifelong; skip parc enqueue")


## 状態確認 / 一時停止 / 再開

- Pause: `pause_job(job_id)`（ckpt は残る）
- Resume: `resume_run(run_id, mode="auto")`（最新 ckpt から eval または train）


In [ ]:
status = queue_status(limit=15)
status if isinstance(status, dict) else print(status)

# JOB_ID = "q_..."
# print(pause_job(JOB_ID))

# RUN_ID = "2026..._smolvla..."
# print(resume_run(RUN_ID, mode="auto"))


## Lifelong（Hydra）を順次実行

途中停止は `pause_lifelong_trial(trial_id)`。次回 `run_lifelong_sweep(..., resume_paused=True)` で `resume_latest.pth` から再開します。


In [ ]:
LL_SWEEP = NOTEBOOKS_DIR / "experiments" / "sweeps" / "lifelong_policy_seed.yaml"
ll = load_sweep(LL_SWEEP)
print(expand_lifelong_trials(ll))

# 実走（長い）:
# run_lifelong_sweep(str(LL_SWEEP), resume_paused=True)

# pause:
# pause_lifelong_trial("lifelong_policy_seed_t000")


## スコア収集とランキング

結果は `notebooks/runs/results.sqlite` に正規化されます。


In [ ]:
print(collect_all(sweep_id=sweep.get("name")))
print_status(sweep_id=sweep.get("name"))
print_ranking(sweep_id=sweep.get("name"))
